In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from pathlib import Path
import time
import math
import copy

PARQUET_DIR = Path.home() / "projects" / "recsys" / "data" / "parquet"
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
torch.manual_seed(42)
np.random.seed(42)
print(f"device: {device}")

ratings = pd.read_parquet(PARQUET_DIR / "ratings_clean.parquet")
cutoff_ts = ratings["timestamp"].quantile(0.9)
train_df_full = ratings[ratings["timestamp"] < cutoff_ts].reset_index(drop=True)
val_df_full   = ratings[ratings["timestamp"] >= cutoff_ts].reset_index(drop=True)

user_to_idx = {u: i for i, u in enumerate(train_df_full["userId"].unique())}
movie_to_idx = {m: i for i, m in enumerate(train_df_full["movieId"].unique())}
n_users = len(user_to_idx)
n_movies = len(movie_to_idx)

def apply_mappings(df):
    df = df.copy()
    df["user_idx"] = df["userId"].map(user_to_idx)
    df["movie_idx"] = df["movieId"].map(movie_to_idx)
    df = df.dropna(subset=["user_idx", "movie_idx"]).reset_index(drop=True)
    df["user_idx"] = df["user_idx"].astype(np.int32)
    df["movie_idx"] = df["movie_idx"].astype(np.int32)
    return df

train_df_full = apply_mappings(train_df_full)
val_df_full   = apply_mappings(val_df_full)

LIKE_THRESHOLD = 4.0
train_pos = train_df_full[train_df_full["rating"] >= LIKE_THRESHOLD].reset_index(drop=True)

print(f"users: {n_users:,} | movies: {n_movies:,}")
print(f"train positives: {len(train_pos):,}")

device: mps
users: 150,330 | movies: 45,058
train positives: 11,177,787


In [2]:
class PopWeightedBPRDataset(Dataset):
    """
    yields (user_idx, pos_movie_idx, neg_movie_idx) triples.
    
    negatives are sampled with probability ∝ popularity^alpha (default 0.75).
    rejection: don't sample movies the user has already interacted with.
    
    why popularity weighting: with uniform sampling, almost every negative is 
    a long-tail obscurity (because there are 45k movies, most with <10 ratings).
    the model learns "popular = positive" — which is just popularity baseline.
    weighting toward popular items forces the model to distinguish between 
    popular-liked and popular-unliked, which is the actual production task.
    """
    def __init__(self, pos_df: pd.DataFrame, all_train_df: pd.DataFrame, 
                 n_movies: int, alpha: float = 0.75, seed: int = 0):
        self.users = torch.from_numpy(pos_df["user_idx"].values.astype(np.int64))
        self.pos_movies = torch.from_numpy(pos_df["movie_idx"].values.astype(np.int64))
        
        # build user_to_seen for rejection
        print("building user_to_seen lookup...")
        t0 = time.time()
        self.user_to_seen = {}
        for user_idx, group in all_train_df.groupby("user_idx"):
            self.user_to_seen[int(user_idx)] = set(group["movie_idx"].values.tolist())
        print(f"  done in {time.time()-t0:.1f}s")
        
        # popularity-weighted sampling distribution
        # pop_count[i] = number of times movie i appears in train (any rating)
        pop_count = np.zeros(n_movies, dtype=np.float64)
        counts = all_train_df["movie_idx"].value_counts()
        pop_count[counts.index.values] = counts.values
        pop_weighted = pop_count ** alpha
        self.sampling_probs = pop_weighted / pop_weighted.sum()
        
        # precompute a big pool of pre-sampled negatives — much faster than 
        # calling np.random.choice per __getitem__
        print(f"pre-sampling negative pool (alpha={alpha})...")
        t0 = time.time()
        rng = np.random.RandomState(seed)
        self.neg_pool = rng.choice(n_movies, size=20_000_000, p=self.sampling_probs).astype(np.int32)
        self.neg_pool_idx = 0
        print(f"  done in {time.time()-t0:.1f}s, pool size: {len(self.neg_pool):,}")
        
        self.n_movies = n_movies
        self.rng = rng
    
    def __len__(self):
        return len(self.users)
    
    def __getitem__(self, i):
        u = int(self.users[i].item())
        pos_m = int(self.pos_movies[i].item())
        seen = self.user_to_seen[u]
        
        # rejection sample from the pre-sampled pool
        for _ in range(20):  # bounded retries
            neg_m = int(self.neg_pool[self.neg_pool_idx % len(self.neg_pool)])
            self.neg_pool_idx += 1
            if neg_m not in seen:
                return self.users[i], self.pos_movies[i], torch.tensor(neg_m, dtype=torch.long)
        # if we somehow can't find an unseen negative in 20 tries, fall back to uniform
        while True:
            neg_m = self.rng.randint(0, self.n_movies)
            if neg_m not in seen:
                return self.users[i], self.pos_movies[i], torch.tensor(neg_m, dtype=torch.long)


# build dataset
train_ds = PopWeightedBPRDataset(train_pos, train_df_full, n_movies, alpha=0.75, seed=42)
print(f"\ntrain dataset: {len(train_ds):,} positive interactions")

# inspect what the sampling distribution looks like
print(f"\nsampling distribution stats:")
print(f"  most-popular movie sample prob: {train_ds.sampling_probs.max()*100:.4f}%")
print(f"  least-popular sample prob:      {train_ds.sampling_probs[train_ds.sampling_probs > 0].min()*100:.6f}%")
print(f"  uniform would be:               {100/n_movies:.4f}%")

# sanity: pull one sample
u, p, n = train_ds[0]
print(f"\nsample 0: user={u.item()}, pos={p.item()}, neg={n.item()}")
print(f"  pos in seen? {p.item() in train_ds.user_to_seen[int(u.item())]}")
print(f"  neg in seen? {n.item() in train_ds.user_to_seen[int(u.item())]}")

building user_to_seen lookup...
  done in 4.6s
pre-sampling negative pool (alpha=0.75)...
  done in 1.2s, pool size: 20,000,000

train dataset: 11,177,787 positive interactions

sampling distribution stats:
  most-popular movie sample prob: 0.1652%
  least-popular sample prob:      0.000036%
  uniform would be:               0.0022%

sample 0: user=0, pos=2, neg=14372
  pos in seen? True
  neg in seen? False


In [3]:
class TwoTower(nn.Module):
    def __init__(self, n_users: int, n_movies: int, dim: int = 64):
        super().__init__()
        self.user_emb = nn.Embedding(n_users, dim)
        self.item_emb = nn.Embedding(n_movies, dim)
        nn.init.normal_(self.user_emb.weight, std=0.01)
        nn.init.normal_(self.item_emb.weight, std=0.01)
    def encode_user(self, u): return self.user_emb(u)
    def encode_item(self, m): return self.item_emb(m)
    def forward(self, u, p, n):
        uv = self.encode_user(u)
        pv = self.encode_item(p)
        nv = self.encode_item(n)
        return (uv * pv).sum(dim=1), (uv * nv).sum(dim=1)


EMB_DIM = 64
BATCH_SIZE = 8192
LR = 0.005
WEIGHT_DECAY = 1e-5
N_EPOCHS = 5

model = TwoTower(n_users, n_movies, dim=EMB_DIM).to(device)
optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)

print(f"params: {sum(p.numel() for p in model.parameters()):,}")
print(f"batches per epoch: {len(train_loader):,}\n")

losses = []
pos_gt_neg = []

for epoch in range(1, N_EPOCHS + 1):
    model.train()
    epoch_loss = 0.0
    epoch_correct = 0
    n_seen = 0
    t0 = time.time()
    
    for u, p, n in train_loader:
        u, p, n = u.to(device), p.to(device), n.to(device)
        optimizer.zero_grad()
        pos_score, neg_score = model(u, p, n)
        loss = -F.logsigmoid(pos_score - neg_score).mean()
        loss.backward()
        optimizer.step()
        
        bsz = len(u)
        epoch_loss += loss.item() * bsz
        epoch_correct += (pos_score > neg_score).sum().item()
        n_seen += bsz
    
    avg_loss = epoch_loss / n_seen
    accuracy = epoch_correct / n_seen
    losses.append(avg_loss)
    pos_gt_neg.append(accuracy)
    
    print(f"epoch {epoch}/{N_EPOCHS} | bpr loss: {avg_loss:.4f} | "
          f"pos>neg: {accuracy*100:.1f}% | {time.time()-t0:.1f}s")

print("\ndone")

params: 12,504,832
batches per epoch: 1,365

epoch 1/5 | bpr loss: 0.5176 | pos>neg: 78.1% | 231.7s
epoch 2/5 | bpr loss: 0.4797 | pos>neg: 82.4% | 225.5s
epoch 3/5 | bpr loss: 0.4756 | pos>neg: 82.9% | 220.1s
epoch 4/5 | bpr loss: 0.4730 | pos>neg: 83.2% | 219.7s
epoch 5/5 | bpr loss: 0.4701 | pos>neg: 83.5% | 219.1s

done


In [4]:
def _dcg(rels):
    return sum(rel / math.log2(i + 2) for i, rel in enumerate(rels))


def evaluate_ranking(model, train_df, val_df, n_movies, k_list=(5, 10, 20),
                     n_sample_users=1000, like_threshold=4.0, seed=42):
    rng = np.random.RandomState(seed)
    
    train_by_user = train_df.groupby("user_idx")["movie_idx"].apply(set)
    val_by_user_liked = (
        val_df[val_df["rating"] >= like_threshold]
        .groupby("user_idx")["movie_idx"].apply(set)
    )
    eligible = list(val_by_user_liked.index)
    print(f"eligible val users: {len(eligible):,}")
    sample_users = rng.choice(eligible, size=min(n_sample_users, len(eligible)), replace=False)
    
    pop_score = np.zeros(n_movies, dtype=np.float32)
    counts = train_df["movie_idx"].value_counts()
    pop_score[counts.index.values] = counts.values
    
    model.eval()
    with torch.no_grad():
        all_movies_t = torch.arange(n_movies, dtype=torch.long, device=device)
        item_vecs = model.encode_item(all_movies_t)
    
    metrics = {f"recall@{k}": [] for k in k_list}
    metrics.update({f"ndcg@{k}": [] for k in k_list})
    metrics.update({f"pop_recall@{k}": [] for k in k_list})
    
    t0 = time.time()
    with torch.no_grad():
        for user_idx in sample_users:
            seen = train_by_user.get(user_idx, set())
            liked = val_by_user_liked[user_idx]
            
            mask = np.ones(n_movies, dtype=bool)
            mask[list(seen)] = False
            
            user_t = torch.tensor([int(user_idx)], dtype=torch.long, device=device)
            user_vec = model.encode_user(user_t)
            scores = (item_vecs @ user_vec.T).squeeze(1).cpu().numpy()
            scores[~mask] = -np.inf
            
            pop_masked = pop_score.copy()
            pop_masked[~mask] = -np.inf
            
            for k in k_list:
                top_model = np.argpartition(-scores, k)[:k]
                hits_model = liked.intersection(top_model.tolist())
                metrics[f"recall@{k}"].append(len(hits_model) / len(liked))
                
                top_sorted = top_model[np.argsort(-scores[top_model])]
                rels = [1 if mid in liked else 0 for mid in top_sorted]
                ideal = [1] * min(k, len(liked))
                ndcg = _dcg(rels) / _dcg(ideal) if ideal else 0
                metrics[f"ndcg@{k}"].append(ndcg)
                
                top_pop = np.argpartition(-pop_masked, k)[:k]
                hits_pop = liked.intersection(top_pop.tolist())
                metrics[f"pop_recall@{k}"].append(len(hits_pop) / len(liked))
    print(f"eval done in {time.time()-t0:.1f}s")
    return {key: float(np.mean(vals)) for key, vals in metrics.items()}


print("evaluating two-tower (pop-weighted bpr) on 1000 val users...")
results = evaluate_ranking(model, train_df_full, val_df_full, n_movies)

# comparison table including all prior models
print(f"\n{'metric':<18s} {'pop-bpr':>10s} {'random-bpr':>12s} {'mf':>10s} {'popularity':>12s}")
print("-" * 66)
prior = {
    "recall@5":  {"random_bpr": 0.0221, "mf": 0.0201, "pop": 0.0202},
    "recall@10": {"random_bpr": 0.0321, "mf": 0.0345, "pop": 0.0301},
    "recall@20": {"random_bpr": 0.0520, "mf": 0.0520, "pop": 0.0468},
}
for k in (5, 10, 20):
    new = results[f"recall@{k}"]
    p = prior[f"recall@{k}"]
    print(f"{'recall@'+str(k):<18s} {new:>10.4f} {p['random_bpr']:>12.4f} {p['mf']:>10.4f} {p['pop']:>12.4f}")
print()
for k in (5, 10, 20):
    print(f"{'ndcg@'+str(k):<18s} {results[f'ndcg@'+str(k)]:>10.4f}")
print(f"\ndelta vs popularity:")
for k in (5, 10, 20):
    d = results[f"recall@{k}"] - prior[f"recall@{k}"]["pop"]
    print(f"  recall@{k}: {d:+.4f}")

evaluating two-tower (pop-weighted bpr) on 1000 val users...
eligible val users: 6,282
eval done in 2.8s

metric                pop-bpr   random-bpr         mf   popularity
------------------------------------------------------------------
recall@5               0.0262       0.0221     0.0201       0.0202
recall@10              0.0434       0.0321     0.0345       0.0301
recall@20              0.0693       0.0520     0.0520       0.0468

ndcg@5                 0.1157
ndcg@10                0.1049
ndcg@20                0.1028

delta vs popularity:
  recall@5: +0.0060
  recall@10: +0.0133
  recall@20: +0.0225


two-tower with popularity-weighted bpr negatives (alpha=0.75, w2v style). recall@10 jumped from 0.0345 (mf) to 0.0434 (+26%), beating popularity baseline by 44%. ndcg@5 from 0.0783 to 0.1157 (+48%). the key insight from this week: the negative sampling distribution is the algorithm. uniform random negatives produce a popularity-shaped model. popularity-weighted negatives force the model to learn user preference. tried in-batch negatives first; training was unstable at small embedding init / small temperature scale on mps. switched to bpr + pop-weighted, which converged cleanly.